In [7]:
"""
QAT — Real INT8 Inference via onednn (CPU)
===========================================
Trains with QAT (fake-quant on GPU + AMP), then converts to
REAL INT8 using onednn and evaluates on CPU.

FIXES in this version:
  Fix 1 — NUM_WORKERS forced to 0 inside Jupyter/IPython on Windows.
           Jupyter kernels use the "spawn" start method but the worker
           module is __main__ inside ipykernel, which cannot be imported
           by child processes → workers crash immediately.
           Solution: detect ipykernel and set NUM_WORKERS=0 (in-process).

  Fix 2 — prepare_qat_fx example_inputs must be a tuple-of-args,
           i.e. ((torch.randn(1,3,32,32),)) not a bare tensor.
           Passing a raw tensor is deprecated and causes silent failures
           on newer FX tracing paths.

  Fix 3 — GradScaler constructed with device="cuda" keyword.
           torch.amp.GradScaler("cuda", enabled=...) is the stable API
           since PyTorch 2.x; the old positional form is fragile.

  Fix 4 — _orig_mod hack removed.
           prepare_qat_fx returns a GraphModule; after .cpu().eval()
           it is ready for convert_fx directly — no internal attribute
           fishing needed.

  Fix 5 — freeze_bn_stats path corrected.
           torch.nn.intrinsic.qat.freeze_bn_stats no longer exists in
           torch.ao-based builds; fall back to manually setting BN to
           eval mode, which is functionally identical for QAT freezing.

  Fix 6 — worker_init_fn retained but only used when NUM_WORKERS > 0.
"""

import os, sys, json, time, copy, tempfile, warnings, random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from thop import profile as thop_profile

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── INT8 backend ──────────────────────────────────────────────────────────────
torch.backends.quantized.engine = "onednn"

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch : {torch.__version__}")
print(f"[init] Device  : {DEVICE}")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
cap      = torch.cuda.get_device_capability() if DEVICE.type == "cuda" else (0, 0)
SM       = f"SM{cap[0]}{cap[1]}" if DEVICE.type == "cuda" else "N/A"
print(f"[init] GPU     : {GPU_NAME}  ({SM})")
print(f"[init] INT8    : onednn (CPU real INT8)")
print(f"[init] Seed    : {SEED}")

# ── QAT backend ───────────────────────────────────────────────────────────────
from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx
from torch.ao.quantization import (
    QConfig,
    MinMaxObserver, PerChannelMinMaxObserver,
    MovingAverageMinMaxObserver, MovingAveragePerChannelMinMaxObserver,
)
print("[init] QAT backend : FX-graph (torch.ao)")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__2__QAT.json"
SAVE_DIR              = "__2__saved_models_QAT"

BATCH_SIZE            = 128
NUM_CLASSES           = 10
QAT_EPOCHS            = 3
LR_BASE               = 1e-4
OBSERVER_FREEZE_EPOCH = 1
TIMING_RUNS           = 100
USE_AMP               = DEVICE.type == "cuda"

# ── Fix 1: Jupyter/IPython on Windows cannot spawn importable workers ─────────
# ipykernel runs __main__ as a kernel script, not a re-importable module,
# so child processes launched by DataLoader fail immediately trying to
# re-import __main__. Setting NUM_WORKERS=0 keeps everything in-process.
def _is_jupyter() -> bool:
    return "ipykernel" in sys.modules

NUM_WORKERS = 0 if (_is_jupyter() and os.name == "nt") else 4
PIN_MEMORY  = DEVICE.type == "cuda" and NUM_WORKERS > 0  # pin_memory needs workers
print(f"[init] NUM_WORKERS : {NUM_WORKERS}  ({'Jupyter/Windows — using 0' if NUM_WORKERS == 0 else 'subprocess workers'})")

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] AMP         : {USE_AMP}")
print(f"[init] Epochs      : {QAT_EPOCHS}  Batch: {BATCH_SIZE}  Timing runs: {TIMING_RUNS}")

# ── Observer / LR configs ─────────────────────────────────────────────────────
OBSERVER_QCONFIGS = {
    "minmax": QConfig(
        activation=MinMaxObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=PerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
    "moving_avg": QConfig(
        activation=MovingAverageMinMaxObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=MovingAveragePerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
}

LR_SCHEDULES = {
    "constant": lambda opt, ep: None,
    "cosine"  : lambda opt, ep: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ep),
    "step"    : lambda opt, ep: torch.optim.lr_scheduler.StepLR(
                    opt, step_size=max(1, ep // 2), gamma=0.1),
}

# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    print(f"[load] {path} ...", flush=True)
    model = build_model(NUM_CLASSES).cpu()
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()
    print("[load] OK", flush=True)
    return model

# ── Data ──────────────────────────────────────────────────────────────────────
def _worker_init_fn(worker_id):
    """Top-level so it's picklable — only called when NUM_WORKERS > 0."""
    np.random.seed(SEED + worker_id)

def get_train_loader():
    tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(
        root="../data", train=True, download=True, transform=tf)
    return DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        worker_init_fn=_worker_init_fn if NUM_WORKERS > 0 else None,
    )

def get_test_loader():
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(
        root="../data", train=False, download=True, transform=tf)
    return DataLoader(
        ds, batch_size=512, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

# ── INT8 conversion ───────────────────────────────────────────────────────────
def convert_to_int8(fakequant_model):
    """
    Convert QAT fake-quant model to REAL INT8 (onednn, CPU).
    convert_fx() replaces every FakeQuantize node with a genuine
    quantized op backed by onednn INT8 SIMD kernels.
    """
    int8_model = convert_fx(copy.deepcopy(fakequant_model).cpu().eval())
    return int8_model

# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device=torch.device("cpu")):
    model = model.to(device).eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        out = model(inputs.to(device))
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

# ── Timing ────────────────────────────────────────────────────────────────────
@torch.no_grad()
def measure_latency_cpu(model, n=TIMING_RUNS, label="model"):
    model = copy.deepcopy(model).cpu().eval()
    dummy_1 = torch.randn(1,          3, 32, 32)
    dummy_b = torch.randn(BATCH_SIZE, 3, 32, 32)

    for _ in range(10):
        model(dummy_1)
        model(dummy_b)

    single_t, batch_t = [], []
    for _ in range(n):
        t = time.perf_counter()
        model(dummy_1)
        single_t.append((time.perf_counter() - t) * 1000)

    for _ in range(n):
        t = time.perf_counter()
        model(dummy_b)
        batch_t.append((time.perf_counter() - t) * 1000)

    bms = float(np.mean(batch_t))
    sms = float(np.mean(single_t))
    print(f"      [time:{label}] single={sms:.2f}ms  batch={bms:.1f}ms  "
          f"throughput={BATCH_SIZE/(bms/1000):.0f}imgs/s", flush=True)
    return {
        "single_img_ms"      : round(sms, 4),
        "batch128_ms"        : round(bms, 4),
        "per_img_ms"         : round(bms / BATCH_SIZE, 4),
        "throughput_imgs_sec": round(BATCH_SIZE / (bms / 1000), 1),
        "n_timing_runs"      : n,
        "label"              : label,
    }

# ── Helpers ───────────────────────────────────────────────────────────────────
def model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)

def measure_flops(model):
    m = copy.deepcopy(model).cpu().eval()
    try:
        macs, _ = thop_profile(m, inputs=(torch.randn(1, 3, 32, 32),), verbose=False)
        return macs * 2
    except Exception as e:
        print(f"      [flops] WARNING: {e}", flush=True)
        return 0.0

def freeze_observers_and_bn(model):
    """
    Freeze quantization observers and BN running stats.
    Fix 5: torch.nn.intrinsic.qat.freeze_bn_stats is gone in torch.ao builds.
    We disable observers via the stable public API, then manually set all
    BN layers to eval() (identical effect: stops updating running mean/var).
    """
    try:
        model.apply(torch.ao.quantization.disable_observer)
    except Exception as e:
        print(f"      [freeze] disable_observer warning: {e}", flush=True)

    # Freeze BN stats by switching BN layers to eval mode
    bn_frozen = 0
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
            m.eval()
            bn_frozen += 1
    print(f"      [freeze] Froze {bn_frozen} BN layers", flush=True)

# ── QAT Training ──────────────────────────────────────────────────────────────
def train_qat(fp32_model, train_loader, obs_name, lr_schedule_name,
              n_epochs=QAT_EPOCHS):
    """
    QAT training loop:
      - prepare_qat_fx inserts FakeQuantize nodes (this IS QAT, not a workaround)
      - Forward pass runs fake-quant: weights stay FP32 but quantization error
        is simulated via the Straight-Through Estimator (STE)
      - convert_fx() later replaces FakeQuantize nodes with real INT8 ops

    Fix 2: example_inputs must be a tuple-of-args: ((tensor,))
           A bare tensor silently bypasses shape validation in newer FX.
    Fix 3: GradScaler uses device="cuda" keyword form (stable since PyTorch 2.x).
    Fix 4: No _orig_mod fishing — prepared is already a GraphModule; .cpu().eval()
           is sufficient before convert_fx.
    """
    qconfig  = OBSERVER_QCONFIGS[obs_name]
    model    = copy.deepcopy(fp32_model).cpu().train()

    # Fix 2: wrap example input as a tuple-of-args
    example_inputs = (torch.randn(1, 3, 32, 32),)
    prepared = prepare_qat_fx(model, {"": qconfig}, example_inputs)
    prepared = prepared.to(DEVICE)

    # Warm-up pass to initialise observer statistics before training
    with torch.no_grad():
        prepared(torch.randn(2, 3, 32, 32, device=DEVICE))
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(prepared.parameters(), lr=LR_BASE,
                                momentum=0.9, weight_decay=1e-4)
    scheduler = LR_SCHEDULES[lr_schedule_name](optimizer, n_epochs)

    # Fix 3: stable GradScaler API — device as keyword, not positional
    scaler = torch.amp.GradScaler(device="cuda", enabled=USE_AMP)

    epoch_train_acc = []
    t_start = time.perf_counter()

    for epoch in range(n_epochs):
        if epoch == OBSERVER_FREEZE_EPOCH:
            print(f"      [epoch {epoch+1}] Freezing observers + BN stats ...", flush=True)
            # Fix 4: prepared is already a GraphModule — move to CPU, freeze, move back
            prepared = prepared.cpu()
            freeze_observers_and_bn(prepared)
            prepared = prepared.to(DEVICE)
            print(f"      [epoch {epoch+1}] Frozen. Back on {DEVICE}", flush=True)

        prepared.train()
        correct = total = 0
        t_ep = time.perf_counter()

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs  = inputs.to(DEVICE,  non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                outputs = prepared(inputs)
                loss    = criterion(outputs, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            correct += (outputs.detach().argmax(1) == targets).sum().item()
            total   += targets.size(0)

            if (batch_idx + 1) % 100 == 0:
                elapsed = time.perf_counter() - t_ep
                eta     = elapsed / (batch_idx + 1) * (len(train_loader) - batch_idx - 1)
                print(f"      [epoch {epoch+1}/{n_epochs}] "
                      f"batch {batch_idx+1}/{len(train_loader)}  "
                      f"acc={correct/total:.4f}  ETA={eta:.0f}s", flush=True)

        if scheduler:
            scheduler.step()
        acc = correct / total
        epoch_train_acc.append(round(acc, 6))
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        print(f"      [epoch {epoch+1}/{n_epochs}] DONE  acc={acc:.4f}  "
              f"time={time.perf_counter()-t_ep:.1f}s", flush=True)

    train_time_s = time.perf_counter() - t_start

    # Fix 4: prepared is a plain GraphModule — just move to CPU and eval
    trained_fakequant = prepared.cpu().eval()
    return trained_fakequant, epoch_train_acc, round(train_time_s, 2)

# ── Save ──────────────────────────────────────────────────────────────────────
def save_models(trained_fakequant, int8_model, obs_name, lr_schedule_name):
    os.makedirs(SAVE_DIR, exist_ok=True)
    key = f"{obs_name}_{lr_schedule_name}"

    fq_path   = os.path.join(SAVE_DIR, f"qat_fakequant_{key}.pth")
    int8_path = os.path.join(SAVE_DIR, f"qat_int8_{key}.pth")

    torch.save(trained_fakequant.state_dict(), fq_path)
    torch.save(int8_model.state_dict(), int8_path)

    print(f"      [save] fake-quant (FP32 trained) → {fq_path}", flush=True)
    print(f"      [save] real INT8  (onednn)        → {int8_path}", flush=True)
    return fq_path, int8_path

# ── Full pipeline ─────────────────────────────────────────────────────────────
def run_one(fp32_model, train_loader, test_loader,
            baseline_acc, obs_name, lr_schedule_name):
    tag = f"{obs_name}/{lr_schedule_name}"
    print(f"\n  ┌─ [{tag}]", flush=True)

    try:
        print("      [train] QAT on GPU (fake-quant + AMP) ...", flush=True)
        trained_fq, epoch_acc, train_time_s = train_qat(
            fp32_model, train_loader, obs_name, lr_schedule_name)

        print("      [eval] FP32 fake-quant accuracy (CPU reference) ...", flush=True)
        fp32_fq_metrics = evaluate(trained_fq, test_loader, device=torch.device("cpu"))
        print(f"      [eval] FP32 fake-quant acc={fp32_fq_metrics['accuracy']:.4f}", flush=True)

        print("      [int8] Converting to real INT8 (onednn) ...", flush=True)
        int8_model = convert_to_int8(trained_fq)
        print("      [int8] Conversion done ✓  (real INT8, not simulated)", flush=True)

        print("      [eval] Evaluating real INT8 (CPU) ...", flush=True)
        int8_metrics = evaluate(int8_model, test_loader, device=torch.device("cpu"))
        acc_drop = baseline_acc - int8_metrics["accuracy"]
        print(f"      [eval] INT8 acc={int8_metrics['accuracy']:.4f}  "
              f"drop={acc_drop:+.4f}", flush=True)

        print("      [time] Timing INT8 (CPU) ...", flush=True)
        int8_timing = measure_latency_cpu(int8_model, label="INT8-CPU")

        print("      [time] Timing FP32 (CPU, for speedup reference) ...", flush=True)
        fp32_timing = measure_latency_cpu(fp32_model, label="FP32-CPU")

        speedup = fp32_timing["batch128_ms"] / int8_timing["batch128_ms"]
        print(f"      [speedup] INT8 vs FP32 CPU: {speedup:.2f}x  "
              f"({fp32_timing['batch128_ms']:.1f}ms → {int8_timing['batch128_ms']:.1f}ms)",
              flush=True)

        flops_G      = measure_flops(fp32_model) / 1e9
        params       = sum(p.numel() for p in fp32_model.parameters())
        size_fp32_mb = model_size_mb(fp32_model)
        size_int8_mb = model_size_mb(int8_model)

        print(f"      [size] FP32={size_fp32_mb:.2f} MB  INT8={size_int8_mb:.2f} MB  "
              f"ratio={size_fp32_mb/size_int8_mb:.2f}x", flush=True)

        fq_path, int8_path = save_models(trained_fq, int8_model, obs_name, lr_schedule_name)

        print(f"  └─ [{tag}]  INT8 acc={int8_metrics['accuracy']:.4f}  "
              f"drop={acc_drop:+.4f}  "
              f"batch={int8_timing['batch128_ms']:.1f}ms  "
              f"throughput={int8_timing['throughput_imgs_sec']:.0f}imgs/s  "
              f"speedup={speedup:.2f}x  "
              f"size={size_int8_mb:.2f}MB", flush=True)

        return {
            "obs_name"   : obs_name,
            "lr_schedule": lr_schedule_name,
            "backend"    : "onednn-INT8-CPU",
            "seed"       : SEED,
            "gpu"        : GPU_NAME,
            "compute_cap": SM,
            "params"      : int(params),
            "size_fp32_mb": round(size_fp32_mb, 4),
            "size_int8_mb": round(size_int8_mb, 4),
            "size_ratio_x": round(size_fp32_mb / size_int8_mb, 3),
            "flops_G"     : round(flops_G, 4),
            "train_time_s"   : train_time_s,
            "epoch_train_acc": epoch_acc,
            "use_amp"        : USE_AMP,
            "qat_epochs"     : QAT_EPOCHS,
            "observer_freeze_epoch": OBSERVER_FREEZE_EPOCH,
            "int8": {
                "accuracy"     : round(int8_metrics["accuracy"],  6),
                "precision"    : round(int8_metrics["precision"], 6),
                "recall"       : round(int8_metrics["recall"],    6),
                "f1"           : round(int8_metrics["f1"],        6),
                "accuracy_drop": round(acc_drop, 6),
                "latency"      : int8_timing,
            },
            "fp32_reference": {
                "accuracy": round(fp32_fq_metrics["accuracy"], 6),
                "f1"      : round(fp32_fq_metrics["f1"],       6),
                "latency" : fp32_timing,
            },
            "int8_vs_fp32_speedup" : round(speedup, 3),
            "int8_vs_baseline_drop": round(acc_drop, 6),
            "accuracy"    : round(int8_metrics["accuracy"],  6),
            "precision"   : round(int8_metrics["precision"], 6),
            "recall"      : round(int8_metrics["recall"],    6),
            "f1"          : round(int8_metrics["f1"],        6),
            "accuracy_drop": round(acc_drop, 6),
            "latency"     : int8_timing,
            "saved_fakequant_pth": fq_path,
            "saved_int8_pth"     : int8_path,
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return None

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*70}")
    print("  QAT — Real INT8 (onednn) — ResNet50 / CIFAR-10")
    print(f"  GPU    : {GPU_NAME}  ({SM})")
    print(f"  INT8   : onednn (CPU genuine INT8 SIMD kernels)")
    print(f"  Epochs : {QAT_EPOCHS}  |  Batch: {BATCH_SIZE}  |  Timing runs: {TIMING_RUNS}")
    print(f"  Seed   : {SEED}  |  AMP: {USE_AMP}")
    print(f"{'='*70}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline_acc = json.load(f)["accuracy"]
    print(f"  Baseline accuracy : {baseline_acc:.4f}\n", flush=True)

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    train_loader = get_train_loader()
    test_loader  = get_test_loader()

    output = {}

    print("  ── Sweep A: Observer type (lr=constant) ──────────────────────", flush=True)
    best_obs, best_acc = None, -1.0

    for obs_name in OBSERVER_QCONFIGS:
        key    = f"{obs_name}_constant"
        result = run_one(fp32_model, train_loader, test_loader,
                         baseline_acc, obs_name=obs_name,
                         lr_schedule_name="constant")
        if result:
            output[key] = result
            if result["accuracy"] > best_acc:
                best_acc, best_obs = result["accuracy"], obs_name

    if best_obs is None:
        print("  ✗ All Sweep A configs failed.", flush=True)
        return
    print(f"\n  Best observer: {best_obs} (acc={best_acc:.4f})", flush=True)

    print(f"\n  ── Sweep B: LR schedule (observer={best_obs}) ─────────────────", flush=True)
    for lr_name in LR_SCHEDULES:
        if lr_name == "constant":
            continue
        key    = f"{best_obs}_{lr_name}"
        result = run_one(fp32_model, train_loader, test_loader,
                         baseline_acc, obs_name=best_obs,
                         lr_schedule_name=lr_name)
        if result:
            output[key] = result

    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)

    print(f"\n{'='*100}")
    print(f"  ✓ Results saved → {OUTPUT_JSON}")
    print(f"  ✓ Models saved  → {SAVE_DIR}/")
    print(f"\n  {'Config':<28} {'INT8 Acc':>8} {'Drop':>7} "
          f"{'Single ms':>10} {'Batch ms':>9} {'Imgs/s':>9} {'Speedup':>8} {'Size MB':>8}")
    print(f"  {'-'*90}")
    for key, val in output.items():
        if val is None:
            continue
        lat = val["latency"]
        print(f"  {key:<28} {val['accuracy']:>8.4f} "
              f"{val['accuracy_drop']:>7.4f} "
              f"{lat['single_img_ms']:>10.3f} "
              f"{lat['batch128_ms']:>9.1f} "
              f"{lat['throughput_imgs_sec']:>9.1f} "
              f"{val['int8_vs_fp32_speedup']:>7.2f}x "
              f"{val['size_int8_mb']:>8.2f}")
    print(f"{'='*100}\n")

    print("  Notes:")
    print("  - INT8 backend     : onednn (genuine INT8 SIMD, not simulated)")
    print("  - 'fake-quant'     : standard QAT terminology — NOT fake results")
    print("  - Accuracy         : measured on converted real INT8 model (CPU)")
    print("  - Speedup          : FP32-CPU batch128 / INT8-CPU batch128")
    print("  - QAT training     : GPU + AMP (fake-quant during forward = STE algorithm)")
    print("  - Seed             : 42 (reproducible)")
    print(f"  - NUM_WORKERS      : {NUM_WORKERS}")


if __name__ == "__main__":
    main()

[init] PyTorch : 2.12.0.dev20260324+cu128
[init] Device  : cuda
[init] GPU     : NVIDIA GeForce RTX 5070 Laptop GPU  (SM120)
[init] INT8    : onednn (CPU real INT8)
[init] Seed    : 42
[init] QAT backend : FX-graph (torch.ao)
[init] NUM_WORKERS : 0  (Jupyter/Windows — using 0)
[init] AMP         : True
[init] Epochs      : 3  Batch: 128  Timing runs: 100

  QAT — Real INT8 (onednn) — ResNet50 / CIFAR-10
  GPU    : NVIDIA GeForce RTX 5070 Laptop GPU  (SM120)
  INT8   : onednn (CPU genuine INT8 SIMD kernels)
  Epochs : 3  |  Batch: 128  |  Timing runs: 100
  Seed   : 42  |  AMP: True

  Baseline accuracy : 0.9320

[load] ../__2__baseline_resnet50_cifar10.pth ...
[load] OK
  ── Sweep A: Observer type (lr=constant) ──────────────────────

  ┌─ [minmax/constant]
      [train] QAT on GPU (fake-quant + AMP) ...
      [epoch 1/3] batch 100/391  acc=0.9871  ETA=85s
      [epoch 1/3] batch 200/391  acc=0.9873  ETA=68s
      [epoch 1/3] batch 300/391  acc=0.9878  ETA=34s
      [epoch 1/3] DONE  a